You can either download this notebook and run it locally, or you can run it in the cloud:<br>

<a href="https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_Image_Visualization_with_itkWidgets.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <br>

[![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_Image_Visualization_with_itkWidgets.ipynb)

# Summary
Interacitve visualization of data is essential to AI research. It is used to validate how the data is being read and pre-processed.  Overlays (labelmaps) on images allow annotations and segmentation results to be evaluated and enable strengths and weaknesses to be identified and documented. Visualization is also critical in communication with collaborators.

[The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/) is a public service funded by the National Cancer Institute which addresses this challenge by providing hosting and de-identification services to take major burdens of data sharing off researchers. Its rich collection of clinical data and annotations is particularly powerful as a community resource when it is paired with interactive code systems, such as Jupyter systems.

TCIA has partnered with the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) to host and distribute our public DICOM data. **This notebook is focused on accessing TCIA's public DICOM data from IDC and visualizing it with itkWidgets.**

# Outline

1. Setup
2. IDC Basics
3. itkWidgets Basics
4. Use Cases
    1. Cinematic volume rendering of images
    2. Visualization of numpy arrays
    3. Visualization of pytorch tensors
    4. Visualization of images and overlays

# 1. Setup

These are the initial steps for running notebooks within various jupyter environments.

In [ ]:
import os
import sys

# Upgrade pip, just in case...
!{sys.executable} -m pip install --upgrade -q pip

In [ ]:
# If running on sagemaker or studiolab, install essential packages and extensions.
if "studio-lab-user" in os.getcwd():
    print("Upgrading dependencies")
    !conda install --yes -q --prefix {sys.prefix} -c conda-forge opencv nodejs

# On many systems you must manually install the imjoy-jupyter-extension!!

If you do not see a blue 'ImJoy' icon on the menu bar in this notebook:
   1) Google CoLab: The following does not apply to Google CoLab - it will not show an ImJoy and all
      should work without modification.
   2) Enable Extensions:  Many Jupyter Lab systems disable jupyter extensions by default,
      and they must be enabled for this notebook to work. Typically this is accomplished using the
      Jupyter interface to select the extension manager (left-hand side, icon that looks like a piece
      of a puzzle) and select the Enable button if it appears.
   2) Install imjoy extension: In the extension manager, search for 'imjoy' and install
      the 'imjoy-jupyter-extension'. The installation can take several minutes. It may also prompt
      you to rebuild, save, and reload your jupyter environment as part of this process.  In the end,
      you should see a blue 'ImJoy' icon on the left side of the menu bar in this notebook.

# 2. IDC Basics

We will use the [**idc-index**](https://github.com/ImagingDataCommons/idc-index) python package to query and download datasets from the NCI Imaging Data Commons.

In [ ]:
# Install idc-index
!{sys.executable} -m pip install --upgrade -q idc-index

In [ ]:
from idc_index import IDCClient
import pandas as pd
import requests

idc_client = IDCClient()

In [ ]:
# Define SeriesInstanceUIDs to download
uids = ["1.3.6.1.4.1.14519.5.2.1.144673070302354240405004153445986965004",
        "1.3.6.1.4.1.14519.5.2.1.66734119033932110513438442707181367414"]

# download the series and return dataframe of metadata
idc_client.download_from_selection(seriesInstanceUID=uids, downloadDir="idc_download")
df = idc_client.index[idc_client.index['SeriesInstanceUID'].isin(uids)]

# display dataframe
display(df)

In [ ]:
# For this demo...

# Install itk for DICOM I/O and for reading DICOM into an itkImage
!{sys.executable} -m pip install --upgrade --pre -q "itk==5.3.0"

# Additionally we'll install numpy and torch to explore a variety of
#    image data structures
!{sys.executable} -m pip install -q torch
!{sys.executable} -m pip install -q numpy

In [ ]:
# Include ITK for DICOM reading
import itk

# Numpy for numpy.arrays
import numpy as np

# Torch for torch.tensors
import torch

In [ ]:
import glob
# Use the data frame to find the Series UID where the Modality is CT
ct_series_uid = df.loc[df['Modality'] == 'CT', 'SeriesInstanceUID'].iloc[0]

# Find the downloaded directory
ct_dir_matches = glob.glob(f"idc_download/**/*/{ct_series_uid}", recursive=True)
if ct_dir_matches:
    dicom_ct_dir = ct_dir_matches[0]

    # Load and sort the DICOM data into a volume
    dicom_image_large = itk.imread(dicom_ct_dir, itk.F)

    # To save time for this demo, we subsample the image in the x and y dimensions
    new_spacing = list(dicom_image_large.GetSpacing())
    new_spacing[:2] = [x*3 for x in new_spacing[:2]]
    new_size = list(dicom_image_large.GetLargestPossibleRegion().GetSize())
    new_size[:2] = [x//3 for x in new_size[:2]]
    dicom_image = itk.resample_image_filter(Input=dicom_image_large,
                                            output_spacing=new_spacing,
                                            output_origin=dicom_image_large.GetOrigin(),
                                            output_direction=dicom_image_large.GetDirection(),
                                            size=new_size)
    print(f"New spacing = {new_spacing}")
    print(f"New size = {new_size}")

# 3. itkWidget Basics

[itkWidgets documentation](https://itkwidgets.readthedocs.io/en/latest/?badge=latest) provides a summary and illustrations of itkWidgets for a wide variety of scientific data visualization use cases.  Here we focus on its application to data on TCIA.

In [ ]:
# This is the installation required for itkWidgets
!{sys.executable} -m pip install --upgrade -q "itkwidgets[all]==1.0a53" imjoy_elfinder

In [ ]:
# This is the most common import command for itkWidgets.
#   The view() function opens an interactive viewer for 2D and 3D
#   data in a variety of formats.
from itkwidgets import view

# 4. Use cases

4.A. Cinematic volume rendering of images<br>
4.B. Visualization of numpy arrays<br>
4.C. Visualization of pytorch tensors<br>
4.D. Visualization of images and overlays<br>

## 4.A. Cinematric volume rendering of images

In [ ]:
viewerA = view(dicom_image)

## 4.B. Visualization of numpy arrays

In [ ]:
# Create an isotropic resampling of the image
new_spacing_iso = [min(dicom_image.GetSpacing())] * 3
new_size_iso = [int(dicom_image.GetLargestPossibleRegion().GetSize()[i] * dicom_image.GetSpacing()[i] / new_spacing_iso[i]) for i in range(3)]
dicom_image_iso = itk.resample_image_filter(Input=dicom_image,
                                            output_spacing=new_spacing_iso,
                                            output_origin=dicom_image.GetOrigin(),
                                            output_direction=dicom_image.GetDirection(),
                                            size=new_size_iso)

# Convert the itk image to a numpy array
dicom_array_iso = itk.GetArrayFromImage(dicom_image_iso)

viewerB = view(dicom_array_iso)

## 4.C. Visualization of pytorch tensors

In [ ]:
# Convert the numpy array to a torch tensor
dicom_tensor = torch.tensor(dicom_array_iso)

viewerC = view(dicom_tensor)

## 4.D. Visualization of images and overlays

Overlays are common in TCIA data, particularly when used for AI research.  For example, overlays (aka labelmaps, aka label_images)
represent segmentations used in training or produced by deep learning systems.  Here we demonstrate the visualization of overlays
using itkWidgets.   For reading overlays from DICOM SEG, RTStruct, and STL files, refer to other notebooks in this repository.

In [ ]:
dicom_threshold_iso = itk.BinaryThresholdImageFilter(Input=dicom_image_iso, LowerThreshold=300, UpperThreshold=1000, OutsideValue=0, InsideValue=1)
arr = itk.GetArrayFromImage(dicom_threshold_iso)

In [ ]:
viewerD = view(image=dicom_image_iso, label_image=dicom_threshold_iso)

In [ ]:
viewerD.set_image_color_map("Grayscale")
viewerD.set_image_color_range([-500,200])
viewerD.set_label_image_blend(0.5)
viewerD.set_view_mode('ZPlane')
#viewerD.set_z_slice(-134)

# Acknowledgements

TCIA is funded by the [Cancer Imaging Program (CIP)](https://imaging.cancer.gov/), a part of the United States [National Cancer Institute (NCI)](https://www.cancer.gov/), and is managed by the [Frederick National Laboratory for Cancer Research (FNLCR)](https://frederick.cancer.gov/).

If you leverage this notebook or any TCIA datasets in your work please be sure to comply with the [TCIA Data Usage Policy](https://wiki.cancerimagingarchive.net/x/c4hF). In particular, make sure to cite the DOI(s) for the specific TCIA datasets you used in addition to TCIA citation provided below!

This notebook was created by [Stephen Aylward (Kitware)](https://www.kitware.com/stephen-aylward/), [Justin Kirby (Frederick National Laboratory for Cancer Research)](https://www.linkedin.com/in/justinkirby82/), [Brianna Major (Kitware)](https://www.kitware.com/brianna-major/), and [Matt McCormick (Kitware)](https://www.kitware.com/matt-mccormick/).   The creation of this notebook was funded, in part, by NIBIB and NIGMS R01EB021396,
NIBIB R01EB014955, NCI R01CA220681, and NINDS R42NS086295.

If you have any questions, suggestions, or issues with itkWidgets, please post them on the [itkwidget issue tracker](https://github.com/InsightSoftwareConsortium/itkwidgets/issues) or feel free to email us at kitware@kitware.com.

## Data Citation
The data used in this notebook was part of the Pediatric-CT-SEG collection:

Jordan, P., Adamson, P. M., Bhattbhatt, V., Beriwal, S., Shen, S., Radermecker, O., Bose, S., Strain, L. S., Offe, M., Fraley, D., Principi, S., Ye, D. H., Wang, A. S., Van Heteren, J., Vo, N.-J., & Schmidt, T. G. (2021). Pediatric Chest/Abdomen/Pelvic CT Exams with Expert Organ Contours (Pediatric-CT-SEG) (Version 2) [Data set]. The Cancer Imaging Archive. https://doi.org/10.7937/TCIA.X0H0-1706

## Publication Citation
The data used in this notebook was part of the Pediatric-CT-SEG collection:

Jordan, P., Adamson, P. M., Bhattbhatt, V., Beriwal, S., Shen, S., Radermecker, O., Bose, S., Strain, L. S., Offe, M., Fraley, D., Principi, S., Ye, D. H., Wang, A. S., Heteren, J., Vo, N., & Schmidt, T. G. (2022). Pediatric chest‐abdomen‐pelvis and abdomen‐pelvis CT images with expert organ contours. In Medical Physics (Vol. 49, Issue 5, pp. 3523–3528). Wiley. https://doi.org/10.1002/mp.15485

## TCIA Citation

Clark, K., Vendt, B., Smith, K., Freymann, J., Kirby, J., Koppel, P., Moore, S., Phillips, S., Maffitt, D., Pringle, M., Tarbox, L., & Prior, F. (2013). The Cancer Imaging Archive (TCIA): Maintaining and Operating a Public Information Repository. Journal of Digital Imaging, 26(6), 1045–1057. https://doi.org/10.1007/s10278-013-9622-7